In [0]:
spark

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col, count, countDistinct

In [0]:
# I am importing Faker to generate fake names
!pip install Faker
from faker import Faker
import random

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 40.2 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
fake = Faker()

In [0]:
num_records = 1000000

In [0]:
# I am creating empty list to store customer data
data = []

In [0]:
# I am creating a function to generate first name and last name
def generate_name():
    first_name = fake.first_name()
    last_name = fake.last_name()
    return first_name, last_name

# I am creating a function to generate email
def generate_email():
    numbers = "0123456789"
    domains = ["gmail.com", "yahoo.com", "icloud.com", "outlook.com"]
    first_name, last_name = generate_name()
    number = ""
    
    for i in range(3):
        # adding one random digit each time
        number += random.choice(numbers)
    
    # selecting one random domain
    domain = random.choice(domains)
    
    # building email using first name, last name, number, and domain
    email = first_name.lower() + last_name.lower() + number + "@" + domain
    
    #returning first name, last name, and email
    return first_name, last_name, email

# generating customer records
for i in range(num_records):
    # generating customer details
    first_name, last_name, email = generate_email()
    # creating one customer row as tuple
    customer = (i, first_name, last_name, email)
    # I am adding row into list
    data.append(customer)


#defining schema for Spark DataFrame
schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True)
])


#creating Spark DataFrame from Python list
df = spark.createDataFrame(data, schema)

# random duplicate fraction between 2% and 20%
frac_value = random.uniform(0.02, 0.2)
# taking a random sample from existing DataFrame to create duplicates
duplicate_data = df.sample(withReplacement=False, fraction=frac_value)
#printing duplicate fraction used
print("\nDuplicate fraction used:", round(frac_value, 2))

#adding duplicate rows to original DataFrame
df = df.union(duplicate_data)


total_records = df.count()
unique_records = df.select(countDistinct("email")).collect()[0][0]
duplicate_records = total_records - unique_records
unique_percentage = (unique_records / total_records) * 100
duplicate_percentage = (duplicate_records / total_records) * 100

print("\nTotal records:", total_records)
print("Unique records:", unique_records)
print("Unique percentage:", round(unique_percentage, 2), "%")
print("Duplicate percentage:", round(duplicate_percentage, 2), "%")
print("\nNumber of duplicated emails:", duplicate_records)



Duplicate fraction used: 0.13

Total records: 2256270
Unique records: 1994517
Unique percentage: 88.4 %
Duplicate percentage: 11.6 %

Number of duplicated emails: 261753


In [0]:
df.display(10)

customer_id,first_name,last_name,email
0,Lucas,Young,lucasyoung376@icloud.com
1,Donna,Gonzales,donnagonzales918@icloud.com
2,Christopher,Schwartz,christopherschwartz808@gmail.com
3,Stephen,Banks,stephenbanks620@gmail.com
4,Jennifer,Arnold,jenniferarnold042@gmail.com
5,Brian,Miller,brianmiller256@gmail.com
6,Phyllis,Ellison,phyllisellison700@yahoo.com
7,Tyler,Hinton,tylerhinton570@yahoo.com
8,William,Gonzalez,williamgonzalez939@gmail.com
9,Bryan,Jones,bryanjones414@yahoo.com
